# T2 — Figure 6 contrast structure  [Reviewer 1 item 1]

Reviewer 1: *"Why does the lymphocyte inflamed panel a to the left and the myeloid dominant vs
immune desert in the middle panel show similar highly differential genes? HLA-DRA, IFI30, etc. at the
very top. This is very confusing and either it is only looking at the immune desert, or the findings
have some flaw."*

The reviewer's first alternative is correct: **both panels use Immune-desert as the reference**, so
the genes they share are the shared immune-presence axis. This notebook quantifies that so the point
can be made with numbers rather than assertion.

## 1. Overlap and nesting of the three adjusted contrasts

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
UP="/mnt/user-data/uploads/Open PBTA/Revision/Week1/_inputs"
d=pd.read_csv(f"{UP}/adjusted_DEG_all_contrasts.tsv",sep="\t")
A="Lymphocyte_inflamed_vs_Immune_desert"; B="Myeloid_dominant_vs_Immune_desert"; C="Lymphocyte_inflamed_vs_Myeloid_dominant"
w=d.pivot(index="gene",columns="contrast",values="adjusted_log2FC")
q=d.pivot(index="gene",columns="contrast",values="q_BH")
sig=lambda c,dirn=+1:set(w.index[(q[c]<0.05)&((w[c]>1) if dirn>0 else (w[c]<-1))])
up={k:sig(k) for k in (A,B,C)}; dn={k:sig(k,-1) for k in (A,B,C)}
print("=== UP (q<0.05, adj log2FC > 1) ===")
for k in (A,B,C): print(f"  {k:42} {len(up[k]):5}   DOWN {len(dn[k])}")

sA,sB=up[A],up[B]
print(f"\nShared by both vs-Immune-desert contrasts : {len(sA&sB):5}")
print(f"Unique to Lymphocyte-inflamed vs Desert   : {len(sA-sB):5}")
print(f"Unique to Myeloid-dominant vs Desert      : {len(sB-sA):5}")
print(f"Jaccard(LI vs ID, MD vs ID)               : {len(sA&sB)/len(sA|sB):.3f}")
print(f"Of the {len(sB)} MD-vs-Desert genes, {len(sA&sB)/len(sB)*100:.1f}% are also LI-vs-Desert")

# is the shared set explained by the LI>MD gradient?
sh=sorted(sA&sB)
sub=w.loc[sh]
print(f"\nAmong the {len(sh)} shared genes: mean adj log2FC  LI vs ID = {sub[A].mean():.3f},  MD vs ID = {sub[B].mean():.3f}")
print(f"  paired Wilcoxon LI-vs-ID > MD-vs-ID : p = {stats.wilcoxon(sub[A],sub[B],alternative='greater').pvalue:.3e}")
print(f"  median ratio of effect sizes        : {np.median(sub[A]/sub[B]):.2f}x")
# orthogonal (Deming) regression slope on ALL genes significant in either
both=sorted(sA|sB); x=w.loc[both,B].values; y=w.loc[both,A].values
xc,yc=x-x.mean(),y-y.mean()
sxx,syy,sxy=(xc**2).mean(),(yc**2).mean(),(xc*yc).mean()
slope=((syy-sxx)+np.sqrt((syy-sxx)**2+4*sxy**2))/(2*sxy)
r=np.corrcoef(x,y)[0,1]
print(f"\nOrthogonal-regression slope (LI-vs-ID on MD-vs-ID), n={len(both)}: {slope:.2f}   Pearson r = {r:.3f}")

# the discriminating contrast
print(f"\nLI vs MD (the contrast that actually separates the two non-desert ecotypes): "
      f"{len(up[C])} up / {len(dn[C])} down")
topC=d[(d.contrast==C)&(d.q_BH<0.05)&(d.adjusted_log2FC>1)].nsmallest(15,"q_BH")["gene"].tolist()
print("  top 15 by q:",", ".join(topC))
print(f"  of these, {sum(g in (sA&sB) for g in topC)}/15 are in the shared vs-Desert set")

rows=[dict(set_name="Shared: LI-vs-Desert AND MD-vs-Desert",n=len(sA&sB)),
      dict(set_name="Unique: LI-vs-Desert only",n=len(sA-sB)),
      dict(set_name="Unique: MD-vs-Desert only",n=len(sB-sA)),
      dict(set_name="LI-vs-Myeloid (discriminating contrast)",n=len(up[C]))]
pd.DataFrame(rows).to_csv("T2_contrast_overlap_summary.tsv",sep="\t",index=False)
pd.DataFrame({"gene":sh,"adj_log2FC_LI_vs_Desert":w.loc[sh,A].round(4),
              "adj_log2FC_MD_vs_Desert":w.loc[sh,B].round(4),
              "adj_log2FC_LI_vs_Myeloid":w.loc[sh,C].round(4)}).to_csv(
    "T2_shared_immune_presence_genes.tsv",sep="\t",index=False)
w.to_pickle("T2_w.pkl"); q.to_pickle("T2_q.pkl")
np.save("T2_sets.npy",np.array([len(sA&sB),len(sA-sB),len(sB-sA)]))


Key numbers:

- **426 of 444 (95.9%)** Myeloid-dominant-vs-Desert genes are **nested inside** the
  Lymphocyte-inflamed-vs-Desert set. Only 18 are unique to it.
- On those 426 shared genes the Lymphocyte-inflamed effect is a median **1.61x larger**
  (paired Wilcoxon P = 7.6e-71).
- Orthogonal-regression slope across all 18,586 genes is **1.96** (Pearson r = 0.80), i.e. the two
  contrasts move along one axis with roughly double the amplitude, not in parallel.
- The contrast that actually separates the two non-desert ecotypes, Lymphocyte-inflamed vs
  Myeloid-dominant, gives 843 up / 113 down, and only **4 of its top 15 genes** are in the shared
  set — the rest (SIGLEC9, CCR5, GPR65, CCL5, KYNU, IL10, CLEC7A) are distinct.

## 2. Revised Figure 6A and new Figure 6C

In [ ]:
import numpy as np, pandas as pd, matplotlib as mpl
mpl.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.patches import Circle
mpl.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.linewidth":0.8,
                     "pdf.fonttype":42,"ps.fonttype":42})
A="Lymphocyte_inflamed_vs_Immune_desert"; B="Myeloid_dominant_vs_Immune_desert"; C="Lymphocyte_inflamed_vs_Myeloid_dominant"
w=pd.read_pickle("T2_w.pkl"); q=pd.read_pickle("T2_q.pkl")
BLUE,RED,GREY,DARK="#3B82F6","#EF4444","#CBD5E1","#7C8798"

# ---------- Figure 6A revised: explicit reference in every panel title ----------
panels=[(A,"Lymphocyte-inflamed  vs  Immune-desert",BLUE),
        (B,"Myeloid-dominant  vs  Immune-desert",RED),
        (C,"Lymphocyte-inflamed  vs  Myeloid-dominant",BLUE)]
fig,ax=plt.subplots(1,3,figsize=(11.4,3.5),dpi=300)
for i,(c,title,col) in enumerate(panels):
    x=w[c].values; y=-np.log10(np.clip(q[c].values,1e-300,None))
    up=(q[c]<0.05)&(w[c]>1); dn=(q[c]<0.05)&(w[c]<-1)
    a=ax[i]
    a.scatter(x[~(up|dn)],y[~(up|dn)],s=1.5,c=GREY,rasterized=True,lw=0)
    a.scatter(x[up],y[up],s=2.2,c=col,rasterized=True,lw=0)
    a.scatter(x[dn],y[dn],s=2.2,c=DARK,rasterized=True,lw=0)
    for v in (-1,1): a.axvline(v,color="k",ls="--",lw=.6)
    a.axhline(-np.log10(.05),color="k",ls="--",lw=.6)
    ref=title.split("  vs  ")[1]
    a.set_title(f"{title}\nreference = {ref}   |   up {int(up.sum())}; down {int(dn.sum())}",
                fontsize=7.5)
    a.set_xlabel("Molecular-group-adjusted log$_2$ fold change")
    if i==0: a.set_ylabel("$-$log$_{10}$(BH q)")
    a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)
fig.suptitle("Molecular-group-adjusted differential expression   log$_2$(TPM+1) ~ ecotype + integrated molecular group",
             fontsize=8.5,y=1.04)
fig.tight_layout()
for e in ("png","pdf"): fig.savefig(f"Figure6A_revised.{e}",dpi=300,bbox_inches="tight")

# ---------- Figure 6C new: the nesting / gradient evidence ----------
sA=set(w.index[(q[A]<0.05)&(w[A]>1)]); sB=set(w.index[(q[B]<0.05)&(w[B]>1)])
sh=sorted(sA&sB); both=sorted(sA|sB)
fig2,ax2=plt.subplots(1,3,figsize=(11.4,3.4),dpi=300)

# C1 nested Euler
a=ax2[0]; a.set_aspect("equal"); a.axis("off")
a.add_patch(Circle((0,0),1.0,fc=BLUE,alpha=.30,ec=BLUE,lw=1.1))
a.add_patch(Circle((.80,0),.34,fc=RED,alpha=.45,ec=RED,lw=1.1))
a.text(-.62,.55,f"Lymphocyte-inflamed\nvs Immune-desert\n{len(sA):,}",ha="center",fontsize=7,color="#1E3A8A")
a.text(.82,-.62,f"Myeloid-dominant\nvs Immune-desert\n{len(sB):,}",ha="center",fontsize=7,color="#7F1D1D")
a.text(.74,0,f"{len(sA&sB)}",ha="center",va="center",fontsize=9,weight="bold")
a.text(1.10,.30,f"{len(sB-sA)}",ha="center",va="center",fontsize=7.5,color="#7F1D1D")
a.text(-.30,0,f"{len(sA-sB):,}",ha="center",va="center",fontsize=9,weight="bold",color="#1E3A8A")
a.set_xlim(-1.15,1.30); a.set_ylim(-1.15,1.25)
a.set_title(f"A  {len(sA&sB)}/{len(sB)} ({len(sA&sB)/len(sB)*100:.0f}%) of Myeloid-dominant genes\n"
            f"are nested inside the Lymphocyte-inflamed set",fontsize=7.5,loc="left")

# C2 effect-size scatter
a=ax2[1]
allg=w.index
x=w.loc[allg,B].values; y=w.loc[allg,A].values
issig=np.array([g in sA or g in sB for g in allg])
a.scatter(x[~issig],y[~issig],s=2,c=GREY,alpha=.35,lw=0,rasterized=True)
a.scatter(x[issig],y[issig],s=3,c=BLUE,alpha=.45,lw=0,rasterized=True)
lim=[-3.0,5.0]; a.plot(lim,lim,"k--",lw=.8,label="identity (y = x)")
xc,yc=x-x.mean(),y-y.mean(); sxx,syy,sxy=(xc**2).mean(),(yc**2).mean(),(xc*yc).mean()
sl=((syy-sxx)+np.sqrt((syy-sxx)**2+4*sxy**2))/(2*sxy); ic=y.mean()-sl*x.mean()
xs=np.linspace(*lim,10); a.plot(xs,sl*xs+ic,color="#0F766E",lw=1.2,label=f"orthogonal fit (slope = {sl:.2f})")
a.set_xlim(lim); a.set_ylim(lim)
a.set_xlabel("adj. log$_2$FC   Myeloid-dominant vs Immune-desert")
a.set_ylabel("adj. log$_2$FC   Lymphocyte-inflamed vs Immune-desert")
a.set_title(f"B  Same axis, larger amplitude (all {len(allg):,} genes)\nslope > 1 means the Desert contrast is a shared gradient",fontsize=7.5,loc="left")
a.legend(fontsize=6.2,frameon=False,loc="upper left")
a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)

# C3 paired effect sizes on the shared genes
a=ax2[2]
sub=w.loc[sh]
parts=a.violinplot([sub[B].values,sub[A].values],positions=[0,1],widths=.75,showextrema=False)
for pc,cc in zip(parts["bodies"],[RED,BLUE]): pc.set_facecolor(cc); pc.set_alpha(.45); pc.set_edgecolor(cc)
a.plot([0,1],[sub[B].median(),sub[A].median()],"ko-",ms=4,lw=1.2)
a.set_xticks([0,1]); a.set_xticklabels(["Myeloid-dominant\nvs Desert","Lymphocyte-inflamed\nvs Desert"],fontsize=7)
a.set_ylabel("adj. log$_2$FC on the 426 shared genes")
a.set_title(f"C  Median {np.median(sub[A]/sub[B]):.2f}$\\times$ larger in Lymphocyte-inflamed\n"
            f"paired Wilcoxon P = 7.6 $\\times$ 10$^{{-71}}$",fontsize=7.5,loc="left")
a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)

fig2.tight_layout(w_pad=2.0)
for e in ("png","pdf"): fig2.savefig(f"Figure6C_new_contrast_structure.{e}",dpi=300,bbox_inches="tight")
print("saved both figures")


## Interpretation

This is a nested gradient, not a duplicated result. Gene counts reproduce the submitted manuscript
exactly (3816/247, 444/72, 843/113), so nothing in the underlying analysis changes — only the
presentation. Three fixes carry the answer:

1. every panel title now names its reference group explicitly;
2. a new panel quantifies the nesting and the ~2x amplitude difference;
3. the discriminating Lymphocyte-inflamed vs Myeloid-dominant contrast is called out in the Results
   text rather than left as the third panel.